# NB06 — Évaluation Complète et Métriques de Génération
> **Phase F** : Validité, Diversité (Levenshtein, Jaccard), Cohérence statistique (KL divergence), et Nouveauté.


In [ ]:
# ==============================================================================
# Cellule 1 : Paramètres (Settings) et Chargement des Données
# ==============================================================================
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT_DIR))
from utils import set_seed, is_valid_sequence, compute_aa_composition, STANDARD_AA

SEED = 42
set_seed(SEED)
K_MER = 3
N_PAIRS = 500  # Nombre de paires pour Levenshtein et Jaccard

train_path = ROOT_DIR / "data" / "train.csv"
if not train_path.exists():
    train_path = ROOT_DIR / "data" / "toy" / "train.csv"
train_seqs = pd.read_csv(train_path)["sequence"].tolist()
print(f"Train réel chargé : {len(train_seqs):,} séquences.")


In [ ]:
# ==============================================================================
# Cellule 2 : Validité Biologique (100% sur 20 AA, Longueurs 100-140)
# ==============================================================================
gen_dir = ROOT_DIR / "results" / "generated"
config_names = ["greedy", "topk10_T08", "nucleus_p95_T10", "nucleus_p80_T12"]

validity_results = {}
generated_datasets = {}

for name in config_names:
    fasta_file = gen_dir / f"{name}.fasta"
    if not fasta_file.exists():
        continue
    seqs = []
    with open(fasta_file, encoding="utf-8") as f:
        for line in f:
            if not line.startswith(">"):
                seqs.append(line.strip())
    generated_datasets[name] = seqs
    n_valid = sum(is_valid_sequence(s) for s in seqs)
    rate = n_valid / max(len(seqs), 1)
    validity_results[name] = rate
    print(f"Config: {name:<18} | Validité: {rate:.1%} ({n_valid}/{len(seqs)})")


In [ ]:
# ==============================================================================
# Cellule 3 : Diversité (Taux d'Unicité et Distance de Levenshtein)
# ==============================================================================
def levenshtein_distance(s1, s2):
    m, n = len(s1), len(s2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1): dp[i][0] = i
    for j in range(n + 1): dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = 0 if s1[i-1] == s2[j-1] else 1
            dp[i][j] = min(dp[i-1][j] + 1, dp[i][j-1] + 1, dp[i-1][j-1] + cost)
    return dp[m][n] / max(m, n)

diversity_results = {}
for name, seqs in generated_datasets.items():
    uniqueness = len(set(seqs)) / max(len(seqs), 1)
    # Calcul Levenshtein sur N_PAIRS paires aléatoires
    idx_pairs = np.random.choice(len(seqs), size=(min(N_PAIRS, len(seqs)), 2))
    dists = [levenshtein_distance(seqs[i], seqs[j]) for i, j in idx_pairs if i != j]
    mean_dist = np.mean(dists) if dists else 0.0
    diversity_results[name] = (uniqueness, mean_dist)
    print(f"Config: {name:<18} | Unicité: {uniqueness:.1%} | Distance Levenshtein moy.: {mean_dist:.3f}")


In [ ]:
# ==============================================================================
# Cellule 4 : Similarité Jaccard sur les 3-mers
# ==============================================================================
def get_kmers(seq, k=3):
    return set(seq[i:i+k] for i in range(len(seq) - k + 1))

def jaccard_similarity(s1, s2, k=3):
    k1, k2 = get_kmers(s1, k), get_kmers(s2, k)
    union = k1 | k2
    return len(k1 & k2) / len(union) if union else 0.0

jaccard_results = {}
for name, seqs in generated_datasets.items():
    idx_pairs = np.random.choice(len(seqs), size=(min(N_PAIRS, len(seqs)), 2))
    sims = [jaccard_similarity(seqs[i], seqs[j], k=K_MER) for i, j in idx_pairs if i != j]
    mean_jaccard = np.mean(sims) if sims else 0.0
    jaccard_results[name] = mean_jaccard
    print(f"Config: {name:<18} | Jaccard {K_MER}-mers moyen : {mean_jaccard:.3f}")


In [ ]:
# ==============================================================================
# Cellule 5 : Cohérence Statistique (Divergence KL sur la composition AA)
# ==============================================================================
real_comp = compute_aa_composition(train_seqs)

def kl_divergence(p_gen, p_real):
    kl = 0.0
    for aa in STANDARD_AA:
        p = max(p_gen.get(aa, 0.0), 1e-8)
        q = max(p_real.get(aa, 0.0), 1e-8)
        kl += p * np.log(p / q)
    return kl

kl_results = {}
for name, seqs in generated_datasets.items():
    gen_comp = compute_aa_composition(seqs)
    kl = kl_divergence(gen_comp, real_comp)
    kl_results[name] = kl
    print(f"Config: {name:<18} | Divergence KL composition : {kl:.4f}")


In [ ]:
# ==============================================================================
# Cellule 6 : Nouveauté vs Train et Tableau de Synthèse Final
# ==============================================================================
train_set = set(train_seqs)
novelty_results = {}

for name, seqs in generated_datasets.items():
    n_new = sum(1 for s in seqs if s not in train_set)
    novelty_results[name] = n_new / max(len(seqs), 1)

# Tableau récapitulatif final
rows = []
for name in config_names:
    if name not in validity_results:
        continue
    rows.append({
        "Configuration": name,
        "Validité": f"{validity_results[name]:.1%}",
        "Unicité": f"{diversity_results[name][0]:.1%}",
        "Levenshtein": f"{diversity_results[name][1]:.3f}",
        "Jaccard 3-mers": f"{jaccard_results[name]:.3f}",
        "KL Compo": f"{kl_results[name]:.4f}",
        "Nouveauté": f"{novelty_results[name]:.1%}"
    })

summary_table = pd.DataFrame(rows)
print("
" + "="*80)
print("TABLEAU DE SYNTHÈSE FINAL — COMPARAISON DES CONFIGURATIONS")
print("="*80)
print(summary_table.to_string(index=False))
print("="*80)
